In [3]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
  "A fully consistent, minimal model for non-linear market impact"
  Donier, Bonart, Mastromatteo, Bouchaud (2015)
  arXiv:1412.0141

  - Stationary LLOB shape  (Eq. 4 / 6)
  - Central self-consistent integral equation for price impact  (Eq. 9)
  - Coefficient A as function of trading rate m0/J  (Eq. 12, Fig. 2 left)
  - Square-root impact I(Q) vs Q for fixed T  (Fig. 2 right)
  - Post-execution impact decay  (Section VI)
  - Order-reversal trajectory  (Section VI)
  - Full PDE simulation of latent order book  (Eq. 7)
  - Asymmetric bid/ask after meta-order  (Eq. 11)
"""

import numpy as np
from scipy import integrate, optimize
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Global style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.family':      'monospace',
    'axes.titlesize':   11,
    'axes.labelsize':   10,
    'legend.fontsize':  8,
})

TEAL   = '#2dd4bf'
ORANGE = '#fb923c'
PURPLE = '#a78bfa'
GREEN  = '#4ade80'
RED    = '#f87171'
YELLOW = '#fbbf24'
BLUE   = '#60a5fa'


# 1.  STATIONARY LLOB SHAPE


def stationary_llob(y, L=1.0):
    """
    Eq. (6): phi_st(y) = -L * y
    Exactly linear latent order book in the LLOB limit (gamma -> 0).
    phi > 0 => net buy pressure; phi < 0 => net sell pressure.
    Price is at y=0 where phi=0.
    """
    return -L * y


def stationary_full(y, lam, nu, D):
    """
    Eq. (4): exponential stationary shape before taking LLOB limit.
    phi_st(y<=0) = (lambda/nu)[1 - exp(gamma*y)], gamma = sqrt(nu/D)
    """
    gamma = np.sqrt(nu / D)
    phi = np.where(y <= 0,
                   (lam / nu) * (1 - np.exp(gamma * y)),
                   -(lam / nu) * (1 - np.exp(-gamma * y)))
    return phi


# 2.  SELF-CONSISTENT EQUATION FOR A  (Eq. 12)


def integrand_A(u, A):
    """Integrand in Eq. (12): exp(-A^2*(1-sqrt(u)) / (4*(1+sqrt(u)))) / sqrt(4*pi*(1-u))"""
    with np.errstate(divide='ignore', invalid='ignore'):
        sq = np.sqrt(np.maximum(u, 0))
        exponent = -A**2 * (1 - sq) / (4 * (1 + sq + 1e-30))
        kern = np.exp(exponent) / np.sqrt(4 * np.pi * np.maximum(1 - u, 1e-15))
    return kern


def rhs_A_equation(A, m0_over_J):
    """RHS of Eq. (12): (m0/J) * integral_0^1 du kern(u, A)"""
    val, _ = integrate.quad(integrand_A, 0, 1, args=(A,), limit=200,
                             epsabs=1e-10, epsrel=1e-10)
    return m0_over_J * val


def solve_A(m0_over_J):
    """Solve A = RHS(A) self-consistently via fixed-point / root finding."""
    if m0_over_J < 1e-6:
        # linear regime: A ~ m0/J * 1/sqrt(pi) * integral = m0/J * sqrt(pi)/... 
        # small m0: A ≈ m0/J * integral_0^1 1/sqrt(4pi(1-u)) du = m0/J * 1
        return m0_over_J  # leading order
    
    def equation(A):
        return A - rhs_A_equation(A, m0_over_J)
    
    # bracket: for large m0/J, A -> sqrt(2*m0/J)
    A_upper = 3 * np.sqrt(m0_over_J) + 5
    try:
        A_sol = optimize.brentq(equation, 1e-12, A_upper, xtol=1e-10, maxiter=500)
    except ValueError:
        A_sol = np.sqrt(2 * m0_over_J)  # fast-regime asymptote
    return A_sol


# 3.  IMPACT DURING META-ORDER  I(Q, T)

def impact_during(Q, T, D=1.0, L=1.0, J=1.0):
    """
    During execution at rate m0 = Q/T:
      y_t = A * sqrt(D*t)  =>  impact at end = A * sqrt(D*T)
    where A solves Eq. (12) with m0/J.
    Returns impact I(Q) = y_T.
    """
    m0 = Q / T
    A = solve_A(m0 / J)
    return A * np.sqrt(D * T)


# 4.  POST-EXECUTION DECAY  (Section VI, linear propagator limit)
def impact_decay_linear(tau, I_T, T, D=1.0):
    """
    Linear propagator post-execution decay (Eq. 10 applied after stop):
      y(T + tau) = (I_T / sqrt(pi)) * [arctan(sqrt(T/tau)) / sqrt(D)]
    But more precisely from Eq.(10) with m=0 for t>T:
      y(T+tau) proportional to integral_0^T ds / sqrt(T+tau-s)
                             = 2*sqrt(T+tau) - 2*sqrt(tau)   (up to factors)
    Normalised: impact(tau) = I_T * [1 - sqrt(tau/(T+tau))] * (correction)
    
    Exact linear propagator: y(t>T) = (m0/L) * integral_0^T ds/sqrt(4piD(t-s))
                                     = (m0/L) * [sqrt(t) - sqrt(t-T)] / sqrt(pi*D)
    """
    t = T + tau
    # From Eq.(10): y(t) = (m0 / (L * sqrt(pi*D))) * (sqrt(t) - sqrt(t - T))  for t > T
    # At t=T: y(T) = (m0 / (L*sqrt(pi*D))) * sqrt(T) = I_T
    # So normalised:
    decay = (np.sqrt(t) - np.sqrt(tau)) / np.sqrt(T)
    return I_T * decay


def impact_decay_nonlinear_pde(tau_arr, T, m0, D=1.0, L=1.0,
                                 Ny=800, y_max=8.0):
    """
    Numerically propagate Eq.(7) PDE after execution stops (m=0 for t>T).
    Returns price trajectory y(T + tau).
    """
    dy = 2 * y_max / Ny
    y = np.linspace(-y_max, y_max, Ny)
    
    # Build initial phi at t=T from self-consistent solution during execution
    A = solve_A(m0 / (L * D**0.5 / np.sqrt(1)))  # m0/J with J=L*sqrt(D) ~ L here
    # Actually J = D*L so m0/J = m0/(D*L)
    A = solve_A(m0 / (D * L))
    
    # phi(y, T) from Eq.(8) with ys = A*sqrt(D*s), integrated numerically
    N_trap = 200
    s_arr = np.linspace(0, T - 1e-6, N_trap)
    phi = -L * y.copy()
    for s in s_arr:
        ys = A * np.sqrt(D * s) if s > 0 else 0.0
        ds = T / N_trap
        kernel = np.exp(-(y - ys)**2 / (4 * D * (T - s + 1e-10))) / np.sqrt(4 * np.pi * D * (T - s + 1e-10))
        phi += m0 * kernel * ds
    
    # Time-step the PDE (explicit FTCS) for tau in tau_arr
    dt = 0.0005
    prices = []
    t_now = 0.0
    tau_idx = 0
    
    r = D * dt / dy**2  # must be < 0.5 for stability
    assert r < 0.5, f"Unstable: r={r:.3f}, reduce dt or increase Ny"
    
    # Laplacian matrix (diffusion only, no source for t>T)
    phi_curr = phi.copy()
    
    tau_targets = list(tau_arr)
    results = []
    
    while tau_idx < len(tau_targets):
        tau_target = tau_targets[tau_idx]
        while t_now < tau_target - dt/2:
            # Diffusion step
            phi_new = phi_curr.copy()
            phi_new[1:-1] = phi_curr[1:-1] + r * (phi_curr[2:] - 2*phi_curr[1:-1] + phi_curr[:-2])
            # Boundary: slope = -L
            phi_new[0]  = phi_new[1]  + L * dy
            phi_new[-1] = phi_new[-2] - L * dy
            phi_curr = phi_new
            t_now += dt
        
        # Find price: where phi changes sign
        sign_changes = np.where(np.diff(np.sign(phi_curr)))[0]
        if len(sign_changes) > 0:
            i = sign_changes[0]
            # Linear interpolation
            y_price = y[i] - phi_curr[i] * dy / (phi_curr[i+1] - phi_curr[i])
        else:
            y_price = 0.0
        results.append(y_price)
        tau_idx += 1
    
    return np.array(results)

# 5.  FULL PDE SIMULATION DURING META-ORDER  (Eq. 7)

def simulate_llob_pde(T_exec, m0, D=1.0, L=1.0, Ny=600, y_max=6.0,
                       record_times=None):
    """
    Simulate the latent order book PDE (Eq. 7) during a buy meta-order.
    Returns: times, price_trajectory, final phi profile, y_grid
    """
    dy = 2 * y_max / Ny
    y = np.linspace(-y_max, y_max, Ny)
    
    # Stable time step
    dt = 0.4 * dy**2 / D
    
    # Initial condition: stationary LLOB
    phi = stationary_llob(y, L)
    
    times = []
    prices = []
    phi_snapshots = {}
    
    t = 0.0
    if record_times is None:
        record_times = [T_exec * f for f in [0.25, 0.5, 0.75, 1.0]]
    record_set = set(record_times)
    
    while t < T_exec + dt/2:
        # Find current price (zero of phi)
        sc = np.where(np.diff(np.sign(phi)))[0]
        if len(sc) > 0:
            i = sc[0]
            y_p = y[i] - phi[i] * dy / (phi[i+1] - phi[i] + 1e-30)
        else:
            y_p = 0.0
        
        times.append(t)
        prices.append(y_p)
        
        for rt in record_times:
            if abs(t - rt) < dt * 0.6 and rt not in phi_snapshots:
                phi_snapshots[rt] = (phi.copy(), y_p)
        
        # PDE update: diffusion + meta-order source at price
        phi_new = phi.copy()
        phi_new[1:-1] += D * dt / dy**2 * (phi[2:] - 2*phi[1:-1] + phi[:-2])
        
        # Add meta-order source: m0 * delta(y - y_price)
        i_p = int((y_p + y_max) / dy)
        if 0 <= i_p < Ny:
            phi_new[i_p] += m0 * dt / dy   # delta approximation
        
        # Boundary: maintain slope -L
        phi_new[0]  = phi_new[1]  + L * dy
        phi_new[-1] = phi_new[-2] - L * dy
        
        phi = phi_new
        t += dt
    
    return np.array(times), np.array(prices), phi_snapshots, y


# 6.  BID/ASK ASYMMETRY AFTER META-ORDER  (Eq. 11)
def compute_bid_ask(phi, y, y_price, q_values, L=1.0):
    """
    Compute effective bid y-(q) and ask y+(q) prices for volume q.
    From Eq.(11): integral_{y-(q)}^{y_price} phi dy = -q  (bid side)
                  integral_{y_price}^{y+(q)} phi dy = -q  (ask side)
    """
    dy = y[1] - y[0]
    i_p = np.searchsorted(y, y_price)
    
    bids, asks = [], []
    for q in q_values:
        # Bid: integrate leftward from y_price until cumulative = q
        cumsum = 0.0
        i_bid = i_p
        while i_bid > 0 and cumsum < q:
            i_bid -= 1
            cumsum += abs(phi[i_bid]) * dy
        y_bid = y[max(i_bid, 0)]
        
        # Ask: integrate rightward from y_price
        cumsum = 0.0
        i_ask = i_p
        while i_ask < len(y) - 1 and cumsum < q:
            cumsum += abs(phi[i_ask]) * dy
            i_ask += 1
        y_ask = y[min(i_ask, len(y)-1)]
        
        bids.append(y_bid)
        asks.append(y_ask)
    
    return np.array(bids), np.array(asks)

# 7.  PLOT ALL FIGURES

def plot_all():
    fig = plt.figure(figsize=(18, 22))
    fig.patch.set_facecolor('#0d1117')
    
    gs = gridspec.GridSpec(3, 3, figure=fig,
                           hspace=0.42, wspace=0.38,
                           left=0.07, right=0.97,
                           top=0.95, bottom=0.05)
    
    ax1 = fig.add_subplot(gs[0, 0])   # Stationary LLOB shape
    ax2 = fig.add_subplot(gs[0, 1])   # A vs m0/J  (Fig.2 left)
    ax3 = fig.add_subplot(gs[0, 2])   # I(Q) vs Q  (Fig.2 right)
    ax4 = fig.add_subplot(gs[1, 0])   # PDE phi snapshots during execution
    ax5 = fig.add_subplot(gs[1, 1])   # Price trajectory during execution
    ax6 = fig.add_subplot(gs[1, 2])   # Post-execution impact decay
    ax7 = fig.add_subplot(gs[2, 0])   # Order reversal trajectory
    ax8 = fig.add_subplot(gs[2, 1])   # Bid/ask asymmetry after meta-order
    ax9 = fig.add_subplot(gs[2, 2])   # Participation rate vs impact scaling

    # ── Fig 1: Stationary LLOB shape ────────────────────────────────────────
    y = np.linspace(-4, 4, 500)
    L = 1.0

    # Full exponential shape (pre-LLOB limit) for several gamma
    for gamma, col, label in [(0.5, PURPLE, r'$\gamma=0.5$'),
                               (1.0, ORANGE, r'$\gamma=1.0$'),
                               (2.0, RED,    r'$\gamma=2.0$')]:
        lam = gamma**2 * L * 1.0  # D=1 => gamma^2=nu/D, J=lam/gamma=L*D => lam=L*D*gamma
        nu  = gamma**2
        phi_full = stationary_full(y, lam=L*nu/gamma, nu=nu, D=1.0)
        ax1.plot(y, phi_full, color=col, lw=1.5, alpha=0.7, label=label)

    # LLOB limit
    phi_lin = stationary_llob(y, L)
    ax1.plot(y, phi_lin, color=TEAL, lw=2.5, ls='--', label=r'LLOB: $\phi=-Ly$')
    ax1.axhline(0, color='#8b949e', lw=0.7, ls=':')
    ax1.axvline(0, color='#8b949e', lw=0.7, ls=':')
    ax1.fill_between(y, phi_lin, 0, where=(y < 0), alpha=0.15, color=TEAL, label='Net bid')
    ax1.fill_between(y, phi_lin, 0, where=(y > 0), alpha=0.15, color=ORANGE, label='Net ask')
    ax1.set_xlabel(r'Price level $y$')
    ax1.set_ylabel(r'$\phi(y) = \rho_B - \rho_A$')
    ax1.set_title('Stationary LLOB Shape\n(Eq. 4 & 6)', pad=8)
    ax1.legend(fontsize=7)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-4, 4)
    ax1.set_ylim(-4, 4)

    # ── Fig 2: A vs m0/J (Fig. 2 left of paper) ─────────────────────────────
    print("Computing A(m0/J) curve...")
    m0_J_range = np.logspace(-2, 3, 60)
    A_vals = np.array([solve_A(x) for x in m0_J_range])

    # Asymptotes
    A_slow = m0_J_range          # slow: A ~ m0/J  (linear)
    A_fast = np.sqrt(2 * m0_J_range)  # fast: A ~ sqrt(2 * m0/J)

    ratio_paper = A_vals / np.sqrt(m0_J_range)  # A / sqrt(m0/J)  -> Fig 2 left y-axis

    ax2.loglog(m0_J_range, ratio_paper, color=TEAL, lw=2.5, label=r'$A/\sqrt{m_0/J}$')
    ax2.loglog(m0_J_range, np.sqrt(m0_J_range), color=ORANGE, lw=1.5, ls='--',
               label=r'Slow: $\sim\sqrt{m_0/J}$')
    ax2.axhline(np.sqrt(2), color=GREEN, lw=1.5, ls=':', label=r'Fast: $\sqrt{2}$')
    ax2.set_xlabel(r'$m_0/J$')
    ax2.set_ylabel(r'$A / \sqrt{m_0/J}$')
    ax2.set_title(r'Fig. 2 Left: $A$ coefficient vs trading rate', pad=8)
    ax2.legend()
    ax2.grid(True, alpha=0.3, which='both')
    ax2.set_xlim(1e-2, 1e3)

    # ── Fig 3: I(Q) vs Q, fixed T (Fig. 2 right of paper) ──────────────────
    print("Computing I(Q) curves...")
    T_fixed = 1.0
    D = 1.0
    L = 1.0
    J = D * L  # = 1.0

    Q_range = np.logspace(-3, 3, 60)
    I_Q_nonlinear = np.array([impact_during(Q, T_fixed, D, L, J) for Q in Q_range])

    # Asymptotes
    I_linear   = Q_range / (J * np.sqrt(np.pi * D * T_fixed))  # small Q: linear
    I_sqrtQ    = np.sqrt(2 * Q_range / J) * (D * T_fixed)**0.25 * 0   # placeholder

    # Normalise by sqrt(D*T)
    norm = np.sqrt(D * T_fixed)
    ax3.loglog(Q_range / J, I_Q_nonlinear / norm, color=TEAL, lw=2.5, label='LLOB non-linear')

    # Linear regime: I ~ Q / (L * sqrt(pi*D*T))
    mask_lin = Q_range < 0.05 * J
    if mask_lin.any():
        ax3.loglog(Q_range[mask_lin] / J,
                   (Q_range[mask_lin] / (J * np.sqrt(np.pi * D * T_fixed))) / norm,
                   color=ORANGE, lw=1.5, ls='--', label=r'Linear: $I \propto Q$')

    # Square-root regime: I ~ sqrt(2Q/J) * D^(1/4)
    mask_sq = Q_range > 2.0 * J
    Q_sq = Q_range[mask_sq]
    I_sq_approx = np.sqrt(2) * (D * Q_sq / J)**0.5
    ax3.loglog(Q_sq / J, I_sq_approx / norm, color=PURPLE, lw=1.5, ls=':',
               label=r'Fast: $I \propto \sqrt{Q}$')

    ax3.set_xlabel(r'$Q/J$ (normalised volume)')
    ax3.set_ylabel(r'$I(Q) / \sqrt{DT}$')
    ax3.set_title('Fig. 2 Right: Impact vs Volume\n(fixed execution time T)', pad=8)
    ax3.legend()
    ax3.grid(True, alpha=0.3, which='both')

    # ── Fig 4: PDE phi snapshots during execution ────────────────────────────
    print("Running PDE simulation...")
    T_exec = 2.0
    m0_sim = 1.5
    D_sim = 1.0
    L_sim = 1.0
    record_t = [0.5, 1.0, 1.5, 2.0]
    
    times_pde, prices_pde, phi_snaps, y_grid = simulate_llob_pde(
        T_exec, m0_sim, D=D_sim, L=L_sim, Ny=500, y_max=5.0,
        record_times=record_t)
    
    colors_snap = [BLUE, TEAL, ORANGE, RED]
    for i, (rt, col) in enumerate(zip(record_t, colors_snap)):
        # Find closest available snapshot
        closest = min(phi_snaps.keys(), key=lambda k: abs(k - rt))
        phi_s, y_p = phi_snaps[closest]
        ax4.plot(y_grid, phi_s, color=col, lw=1.5, label=f't={rt:.1f}')
        ax4.axvline(y_p, color=col, lw=0.8, ls=':')
    
    # Initial state
    ax4.plot(y_grid, stationary_llob(y_grid, L_sim), color='#8b949e',
             lw=1.2, ls='--', label='t=0 (initial)')
    ax4.axhline(0, color='#30363d', lw=0.6)
    ax4.set_xlim(-4, 4)
    ax4.set_ylim(-4, 4)
    ax4.set_xlabel(r'Price level $y$')
    ax4.set_ylabel(r'$\phi(y,t)$')
    ax4.set_title('PDE: LLOB Deformation\nDuring Buy Meta-Order (Eq. 7)', pad=8)
    ax4.legend(fontsize=7)
    ax4.grid(True, alpha=0.3)

    # ── Fig 5: Price trajectory y(t) = A*sqrt(D*t) during execution ─────────
    ax5.plot(times_pde, prices_pde, color=TEAL, lw=2, label='PDE simulation', alpha=0.9)
    
    # Theoretical: y_t = A * sqrt(D*t)
    A_th = solve_A(m0_sim / (D_sim * L_sim))
    t_th = np.linspace(0.01, T_exec, 200)
    y_th = A_th * np.sqrt(D_sim * t_th)
    ax5.plot(t_th, y_th, color=ORANGE, lw=2, ls='--',
             label=fr'Theory: $A\sqrt{{Dt}}$, $A$={A_th:.3f}')
    ax5.set_xlabel(r'Time $t$')
    ax5.set_ylabel(r'Price impact $y_t$')
    ax5.set_title(r'Price Trajectory During Meta-Order''\n'r'$y_t = A\sqrt{Dt}$ (Section V)', pad=8)
    ax5.legend()
    ax5.grid(True, alpha=0.3)

    # ── Fig 6: Post-execution impact decay ───────────────────────────────────
    print("Computing post-execution decay...")
    T_decay = 1.0
    tau_arr = np.linspace(1e-3, 5 * T_decay, 200)
    
    for m0_d, col, lbl in [(0.5, TEAL, r'$m_0/J=0.5$ (slow)'),
                            (2.0, ORANGE, r'$m_0/J=2$ (medium)'),
                            (8.0, RED, r'$m_0/J=8$ (fast)')]:
        A_d = solve_A(m0_d / (D * L))
        I_T = A_d * np.sqrt(D * T_decay)
        
        # Linear propagator post-decay: y(T+tau)/I_T = (sqrt(T+tau)-sqrt(tau))/sqrt(T)
        decay_lin = (np.sqrt(T_decay + tau_arr) - np.sqrt(tau_arr)) / np.sqrt(T_decay)
        ax6.plot(tau_arr / T_decay, decay_lin, color=col, lw=2, label=lbl)
    
    # Power-law decay reference: ~1/sqrt(tau)
    tau_ref = tau_arr[tau_arr > 0.1]
    ax6.plot(tau_ref / T_decay, 0.8 / np.sqrt(tau_ref / T_decay + 1),
             color=PURPLE, lw=1.5, ls=':', label=r'$\sim 1/\sqrt{\tau}$ decay')
    
    ax6.axhline(0, color='#8b949e', lw=0.7, ls=':')
    ax6.set_xlabel(r'$\tau / T$')
    ax6.set_ylabel(r'$y(T+\tau) / I(T)$')
    ax6.set_title('Post-Execution Impact Decay\n(Section VI)', pad=8)
    ax6.legend(fontsize=7)
    ax6.grid(True, alpha=0.3)
    ax6.set_xlim(0, 5)

    # ── Fig 7: Order reversal trajectory ────────────────────────────────────
    print("Computing order reversal...")
    # Strategy: buy for t in [0,T], sell for t in [T, 2T]
    # Using linear propagator (Eq.10) for tractability
    T_rev = 1.0
    m0_buy  =  2.0
    m0_sell = -2.0
    t_arr = np.linspace(0.01, 3 * T_rev, 300)
    
    def y_linear_propagator(t, T_buy, T_total, m_buy, m_sell, D=1.0, L=1.0):
        """Linear propagator y(t) = (1/L) * integral_0^t ds m(s)/sqrt(4piD(t-s))"""
        eps = 1e-8
        def integrand(s, t_now):
            if t_now <= s + eps:
                return 0.0
            if s < T_buy:
                m_s = m_buy
            elif s < T_total:
                m_s = m_sell
            else:
                m_s = 0.0
            return m_s / np.sqrt(4 * np.pi * D * (t_now - s + eps))
        
        result, _ = integrate.quad(integrand, 0, min(t, T_total), args=(t,),
                                    limit=100, epsabs=1e-8)
        return result / L
    
    T_total = 2 * T_rev
    y_rev = np.array([y_linear_propagator(t, T_rev, T_total, m0_buy, m0_sell)
                      for t in t_arr])
    
    ax7.plot(t_arr, y_rev, color=TEAL, lw=2.5, label='Buy to Sell reversal')
    ax7.axvline(T_rev, color=ORANGE, lw=1.2, ls='--', alpha=0.8, label='Switch buy to sell')
    ax7.axvline(2*T_rev, color=RED, lw=1.2, ls='--', alpha=0.8, label='End of sell')
    ax7.axhline(0, color='#8b949e', lw=0.7, ls=':')
    
    # Shade regions
    t_buy  = t_arr[t_arr <= T_rev]
    t_sell = t_arr[(t_arr > T_rev) & (t_arr <= 2*T_rev)]
    t_post = t_arr[t_arr > 2*T_rev]
    ax7.fill_between(t_buy,  y_rev[:len(t_buy)],  0, alpha=0.15, color=TEAL)
    ax7.fill_between(t_sell, y_rev[len(t_buy):len(t_buy)+len(t_sell)], 0,
                     alpha=0.15, color=RED)
    ax7.fill_between(t_post, y_rev[len(t_buy)+len(t_sell):], 0, alpha=0.1, color=PURPLE)
    
    ax7.set_xlabel(r'Time $t/T$')
    ax7.set_ylabel(r'Price impact $y_t$')
    ax7.set_title('Order Reversal: Buy then Sell\n(Section VI, Eq. 10)', pad=8)
    ax7.legend(fontsize=8)
    ax7.grid(True, alpha=0.3)

    # ── Fig 8: Bid/Ask asymmetry after meta-order ────────────────────────────
    # Use final PDE snapshot to compute asymmetric bid/ask
    closest_final = min(phi_snaps.keys(), key=lambda k: abs(k - T_exec))
    phi_final, y_price_final = phi_snaps[closest_final]
    
    q_vals = np.linspace(0.01, 2.0, 50)
    bids, asks = compute_bid_ask(phi_final, y_grid, y_price_final, q_vals, L=L_sim)
    
    # Equilibrium (symmetric): y±(q) = y_p ± sqrt(2q/L)
    eq_spread = np.sqrt(2 * q_vals / L_sim)
    
    ax8.plot(q_vals, asks - y_price_final, color=RED, lw=2, label='Ask spread (post meta-order)')
    ax8.plot(q_vals, y_price_final - bids, color=TEAL, lw=2, label='Bid spread (post meta-order)')
    ax8.plot(q_vals, eq_spread, color='#8b949e', lw=1.5, ls='--', label=r'Equilibrium: $\sqrt{2q/L}$')
    
    ax8.set_xlabel(r'Volume $q$')
    ax8.set_ylabel(r'Spread from mid-price')
    ax8.set_title('Bid/Ask Asymmetry After Buy Meta-Order\n(Eq. 11 — depleted ask side)', pad=8)
    ax8.legend(fontsize=8)
    ax8.grid(True, alpha=0.3)

    # ── Fig 9: Participation rate vs impact pre-factor scaling ───────────────
    # Show A*sqrt(D*T) / sqrt(Q) as function of participation rate eta = m0/J
    eta_range = np.logspace(-2, 3, 80)
    A_range = np.array([solve_A(e) for e in eta_range])
    
    # I(Q,T) / sqrt(Q) = A*sqrt(D*T) / sqrt(m0*T) = A / sqrt(m0/D) 
    # Normalise: A / sqrt(eta) * (D/J)^(1/2) -- show shape
    scaling = A_range / np.sqrt(eta_range)  # converges to sqrt(2) for large eta
    
    ax9.semilogx(eta_range, scaling, color=TEAL, lw=2.5,
                 label=r'$A(\eta)/\sqrt{\eta}$, $\eta=m_0/J$')
    ax9.axhline(np.sqrt(2), color=GREEN, lw=1.5, ls='--',
                label=r'Fast limit: $\sqrt{2}$')
    ax9.axhline(0, color='#30363d', lw=0.5)
    
    # Mark crossover region
    ax9.axvspan(0.3, 3.0, alpha=0.08, color=YELLOW, label='Crossover region')
    ax9.set_xlabel(r'Participation rate $\eta = m_0/J$')
    ax9.set_ylabel(r'Impact pre-factor $A/\sqrt{\eta}$')
    ax9.set_title(r'Impact Pre-factor vs Participation Rate''\n(Slow to Fast crossover)', pad=8)
    ax9.legend(fontsize=8)
    ax9.grid(True, alpha=0.3, which='both')
    ax9.set_ylim(0, 2.5)

    # ── Super title ──────────────────────────────────────────────────────────
    fig.text(0.5, 0.975,
             'Replication: Donier, Bonart, Mastromatteo, Bouchaud (2015)\n'
             '"A fully consistent, minimal model for non-linear market impact"  '
             '[arXiv:1412.0141]',
             ha='center', va='top', fontsize=11,
             color='#e6edf3', fontfamily='monospace')

    # To this:
    plt.savefig('llob_replication.png',
            dpi=160, bbox_inches='tight', facecolor='#0d1117')
    print("Saved.")
    plt.close()


if __name__ == '__main__':
    print("Replicating Donier et al. (2015) LLOB model...")
    plot_all()
    print("Done.")

Replicating Donier et al. (2015) LLOB model...
Computing A(m0/J) curve...
Computing I(Q) curves...
Running PDE simulation...
Computing post-execution decay...
Computing order reversal...
Saved.
Done.
